# Train NNUE on Lichess positions plus self-play games of the engine (Kaggle)

This notebook does everything by itself with **Save & Run All** (GPU and Internet on, and the self-play dataset added as input):

1. Quick check of the whole pipeline on a small sample, and a check that the self-play data is found (fails within minutes if something is wrong)
2. Streams all quiet, deeply evaluated positions from the Lichess evaluation database (about 143 million positions, the same data as notebook 05)
3. Adds the **self-play positions**: games the engine played against itself on the laptop (`engine-rs.exe datagen`). Every position has the score of the engine's own search **and the result of the game**
4. Trains the network for as many epochs as fit in the time budget
5. Writes `nnue.bin` (format NNU4), ready for the Rust engine, and prints the piece values the network learned

The network has the same shape as notebook 05 (768 -> 1024x2 -> 1 with SCReLU). Only the data changes, so a match against the notebook 05 net shows what the self-play data is worth.

**Why the game result?** The Lichess data only has a Stockfish score. A score says "white is a bit better", but not whether that advantage actually wins. In the self-play data the target is a mix: `(1 - wdl_weight) * score + wdl_weight * result`. Positions with an advantage that keeps winning (for example a passed pawn) get a higher target than positions where the advantage disappears.

**Before running:** upload `selfplay.bin` as a Kaggle dataset and add it to this notebook with **Add Input**. Every file that matches `selfplay*.bin` under `/kaggle/input` is used.

In [ ]:
!pip install -q zstandard

In [ ]:
import collections
import glob
import json
import math
import multiprocessing
import os
import struct
import time
import urllib.request

import numpy as np
import torch
import zstandard
from matplotlib import pyplot as plt
from torch import nn

## Settings

In [ ]:
notebook_start = time.time()

max_positions = 150_000_000
max_load_minutes = 240
min_depth = 25
val_size = 2_000_000
test_size = 2_000_000
batch_size = 4096
gpu_data_fraction = 0.6

num_hidden = 1024

learning_rate = 0.002

selfplay_pattern = "/kaggle/input/**/selfplay*.bin"
wdl_weight = 0.3

max_epochs = 40
total_hours = 8
reserve_minutes = 30

CP_SCALE = 0.00184104

url = "https://database.lichess.org/lichess_db_eval.jsonl.zst"
out_dir = "/kaggle/working"
log_file = os.path.join(out_dir, "nnue_selfplay.log")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
num_workers = os.cpu_count()

print("device:", device, "| CPU cores:", num_workers)

## Helpers

In [ ]:
def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()

## Loading the data: streamed from Lichess, quiet positions with a deep evaluation

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    index = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            index += int(char)
        else:
            row[index ^ 56] = PIECE_CODES[char]
            index += 1
    return row, color == "w"


def square_index(square):
    return (int(square[1]) - 1) * 8 + (ord(square[0]) - ord("a"))


def is_quiet(row, move):
    if len(move) == 5:
        return False
    from_square, to_square = square_index(move[:2]), square_index(move[2:4])
    if row[to_square] != 0:
        return False
    if row[from_square] in (1, 7) and move[0] != move[2]:
        return False
    return True


def score_to_label(cp, mate):
    if mate is not None:
        return 1.0 if mate > 0 else -1.0
    return 2 / (1 + math.exp(-0.00368208 * cp)) - 1


def parse_chunk(chunk):
    rows, sides, values = [], [], []
    num_lines = 0
    for line in chunk.splitlines():
        num_lines += 1
        data = json.loads(line)
        if not data["evals"]:
            continue
        deepest = max(data["evals"], key=lambda e: e["depth"])
        pv = deepest["pvs"][0]
        if deepest["depth"] < min_depth or not pv.get("line"):
            continue
        row, white = parse_fen(data["fen"])
        if not is_quiet(row, pv["line"].split()[0]):
            continue
        rows.append(row[0::2] | (row[1::2] << 4))
        sides.append(white)
        values.append(score_to_label(pv.get("cp"), pv.get("mate")))
    packed = np.array(rows, dtype=np.uint8).reshape(-1, 32)
    return packed, np.array(sides, dtype=np.uint8), np.array(values, dtype=np.float16), num_lines


def read_chunks(reader, chunk_size=16_000_000):
    rest = b""
    while True:
        block = reader.read(chunk_size)
        if not block:
            break
        block = rest + block
        end = block.rfind(b"\n") + 1
        rest = block[end:]
        yield block[:end]
    if rest:
        yield rest


def load_positions(max_positions, max_minutes):
    packed = np.zeros((max_positions, 32), dtype=np.uint8)
    side = np.zeros(max_positions, dtype=np.uint8)
    labels = np.zeros(max_positions, dtype=np.float16)
    count, num_lines, next_log = 0, 0, 5_000_000
    start = time.time()

    with urllib.request.urlopen(url) as response, multiprocessing.get_context("fork").Pool(num_workers) as pool:
        chunks = read_chunks(zstandard.ZstdDecompressor().stream_reader(response))
        pending = collections.deque()
        while count < max_positions and time.time() - start < max_minutes * 60:
            while len(pending) < 2 * num_workers:
                chunk = next(chunks, None)
                if chunk is None:
                    break
                pending.append(pool.apply_async(parse_chunk, (chunk,)))
            if not pending:
                break

            chunk_packed, chunk_side, chunk_labels, chunk_lines = pending.popleft().get()
            size = min(len(chunk_labels), max_positions - count)
            packed[count:count + size] = chunk_packed[:size]
            side[count:count + size] = chunk_side[:size]
            labels[count:count + size] = chunk_labels[:size]
            count += size
            num_lines += chunk_lines
            if count >= next_log:
                log(f"Loaded {count:,} / {max_positions:,} positions from {num_lines:,} lines")
                next_log += 5_000_000

    log(f"Done: {count:,} positions from {num_lines:,} lines in {(time.time() - start) / 60:.0f} min")
    return packed[:count], side[:count], labels[:count]

## Loading the self-play data

Each record is 36 bytes: the board (32 bytes, two squares per byte, the same packing as the Lichess positions), the side to move (1 byte), the engine's score in centipawns for white (2 bytes) and the result of the game (1 byte: 0 black won, 1 draw, 2 white won).

The engine only wrote positions where the side to move is not in check and the best move is quiet, just like the Lichess filter.

In [ ]:
SELFPLAY_RECORD = np.dtype([("board", "u1", 32), ("white", "u1"), ("score", "<i2"), ("result", "u1")])


def load_selfplay(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    assert files, f"no self-play files found for {pattern}: add the self-play dataset with Add Input"
    records = np.concatenate([np.fromfile(path, dtype=SELFPLAY_RECORD) for path in files])
    score_labels = 2 / (1 + np.exp(-0.00368208 * records["score"].astype(np.float32))) - 1
    result_labels = records["result"].astype(np.float32) - 1
    labels = (1 - wdl_weight) * score_labels + wdl_weight * result_labels
    log(f"Self-play: {len(records):,} positions from {len(files)} file(s), results (black win, draw, white win): {np.bincount(records['result'], minlength=3) / len(records)}")
    return np.ascontiguousarray(records["board"]), records["white"].copy(), labels.astype(np.float16)

## Features: 768 piece-square inputs per perspective

Each perspective sees the board from its own side: for black the board is flipped, so "own pieces" and "enemy pieces" always mean the same thing.

In [ ]:
PIECE_TYPES = torch.tensor([0, 0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], device=device)
PIECE_COLORS = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], device=device)
SQUARES = torch.arange(64, device=device)
NUM_FEATURES = 768


def unpack(data_packed):
    low = data_packed & 15
    high = data_packed >> 4
    return torch.stack([low, high], dim=2).reshape(-1, 64).long()


def perspective_features(board, perspective):
    own = (PIECE_COLORS[board] != perspective).long() * 384
    pieces = own + PIECE_TYPES[board] * 64 + (SQUARES ^ (56 * perspective))
    return pieces.masked_fill(board == 0, NUM_FEATURES).sort(dim=1).values[:, :32]


def features(board, white):
    x_white = perspective_features(board, 0)
    x_black = perspective_features(board, 1)
    white = white.bool().unsqueeze(1)
    return torch.where(white, x_white, x_black), torch.where(white, x_black, x_white)

## Model

SCReLU: the hidden values are clipped between 0 and 1 and then squared. Weak activations count less and strong ones count more. In the engine this costs one extra multiplication per hidden unit.

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_hidden):
        super().__init__()
        bound = 1 / math.sqrt(NUM_FEATURES)
        self.feature_transformer = nn.EmbeddingBag(NUM_FEATURES + 1, num_hidden, mode="sum", padding_idx=NUM_FEATURES)
        self.feature_bias = nn.Parameter(torch.empty(num_hidden).uniform_(-bound, bound))
        self.output = nn.Linear(in_features=2 * num_hidden, out_features=1)
        nn.init.uniform_(self.feature_transformer.weight, -bound, bound)

    def forward(self, x_us, x_them):
        us = torch.clamp(self.feature_transformer(x_us) + self.feature_bias, 0, 1) ** 2
        them = torch.clamp(self.feature_transformer(x_them) + self.feature_bias, 0, 1) ** 2
        x = torch.cat([us, them], dim=1)
        return self.output(x)[:, 0]


def build_model(num_hidden):
    return NNUE(num_hidden=num_hidden).to(device)

## Engine file (nnue.bin, format NNU4) and a Python version of the engine evaluation

The Rust engine reads this file. Its evaluation was checked against this Python version on 2,000 positions (largest difference 0.000006) before this notebook was made.

In [ ]:
ENGINE_PARTS = ["feature_weights", "feature_bias", "output_weight", "output_bias"]


def write_nnue(net, path):
    with open(path, "wb") as f:
        f.write(b"NNU4")
        f.write(struct.pack("<3I", NUM_FEATURES, net["num_hidden"], net["screlu"]))
        for name in ENGINE_PARTS:
            f.write(np.asarray(net[name], dtype="<f4").tobytes())


def read_nnue(path):
    with open(path, "rb") as f:
        data = f.read()
    assert data[:4] == b"NNU4", "not an NNU4 file"
    num_features, num_hidden, screlu = struct.unpack("<3I", data[4:16])
    net = {"num_hidden": num_hidden, "screlu": screlu}
    shapes = {
        "feature_weights": (num_features, num_hidden),
        "feature_bias": (num_hidden,),
        "output_weight": (1, 2 * num_hidden),
        "output_bias": (1,),
    }
    values = np.frombuffer(data, dtype="<f4", offset=16)
    start = 0
    for name in ENGINE_PARTS:
        size = math.prod(shapes[name])
        net[name] = values[start:start + size].reshape(shapes[name])
        start += size
    assert start == len(values), "unexpected file size"
    return net


def reference_features(row, perspective):
    flip = 56 * perspective
    active = []
    for square in range(64):
        code = int(row[square])
        if code == 0:
            continue
        own = 0 if (code <= 6) == (perspective == 0) else 384
        active.append(own + (code - 1) % 6 * 64 + (square ^ flip))
    return active


def reference_eval(net, row, white):
    us, them = (0, 1) if white else (1, 0)
    hidden = []
    for perspective in (0, 1):
        active = reference_features(row, perspective)
        values = np.clip(net["feature_bias"] + net["feature_weights"][active].sum(axis=0), 0, 1)
        hidden.append(values ** 2 if net["screlu"] else values)
    x = np.concatenate([hidden[us], hidden[them]])
    return float((net["output_weight"] @ x + net["output_bias"])[0])

In [ ]:
def model_weights(model):
    return {name: value.detach().cpu().numpy() for name, value in model.state_dict().items()}


def net_from_model(model):
    weights = model_weights(model)
    return {
        "num_hidden": weights["feature_bias"].shape[0],
        "screlu": 1,
        "feature_weights": weights["feature_transformer.weight"][:NUM_FEATURES],
        "feature_bias": weights["feature_bias"],
        "output_weight": weights["output.weight"],
        "output_bias": weights["output.bias"],
    }


def export_model(model, path):
    write_nnue(net_from_model(model), path)


def predict(model, row, white):
    board = torch.from_numpy(np.asarray(row)).long().unsqueeze(0).to(device)
    data_white = torch.tensor([white], device=device)
    with torch.no_grad():
        x_us, x_them = features(board, data_white)
        return model(x_us, x_them).item()

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def batches(data, indices, shuffle):
    data_packed, data_side, data_labels = data
    if shuffle:
        indices = indices[torch.randperm(len(indices), device=indices.device)]
    for start in range(0, len(indices), batch_size):
        batch = indices[start:start + batch_size].long()
        board = unpack(data_packed[batch].to(device, non_blocking=True))
        white = data_side[batch].to(device, non_blocking=True).bool()
        x_us, x_them = features(board, white)
        labels = data_labels[batch].to(device, non_blocking=True).float()
        targets = torch.where(white, labels, -labels)
        yield x_us, x_them, targets


def train_one_epoch(model, optimizer, data, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, targets in batches(data, indices, shuffle=True):
        preds = torch.tanh(model(x_us, x_them))
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach()
        num_batches += 1

    return (total_loss / num_batches).item()


def eval_model(model, data, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, targets in batches(data, indices, shuffle=False):
            preds = torch.tanh(model(x_us, x_them))

            total_loss += loss_fn(preds, targets)
            num_batches += 1

            total_abs_error += (preds - targets).abs().sum()
            num_preds += targets.shape[0]

    return (total_loss / num_batches).item(), (total_abs_error / num_preds).item()


def measure_speed(learning_rate, data, indices, num_batches=300, warmup=50):
    model = build_model(num_hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    for i, (x_us, x_them, targets) in enumerate(batches(data, indices, shuffle=True)):
        if i == warmup:
            synchronize()
            start = time.time()
        if i == num_batches:
            break
        loss = loss_fn(torch.tanh(model(x_us, x_them)), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    synchronize()
    return (num_batches - warmup) * batch_size / (time.time() - start)

## Quick check of the whole pipeline

In [ ]:
log("Quick check of the whole pipeline on a small sample...")
sample = tuple(torch.from_numpy(array).to(device) for array in load_positions(20_000, max_minutes=10))
sample_idx = torch.arange(len(sample[2]), device=device, dtype=torch.int32)

check_model = build_model(32)
check_optimizer = torch.optim.Adam(check_model.parameters(), lr=1e-3)
train_loss = train_one_epoch(check_model, check_optimizer, sample, sample_idx)
val_loss, val_mae = eval_model(check_model, sample, sample_idx)
log(f"Small model: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}")
log(f"Speed of the full model: {measure_speed(1e-3, sample, sample_idx, num_batches=4, warmup=1):,.0f} positions/s")

with torch.no_grad():
    for param in check_model.parameters():
        param.add_(0.01 * torch.randn_like(param))
check_path = os.path.join(out_dir, "check.bin")
export_model(check_model, check_path)
check_net = read_nnue(check_path)
os.remove(check_path)

feature_errors, max_error = 0, 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    board = torch.from_numpy(row).unsqueeze(0).to(device)
    x_us, x_them = features(board, torch.tensor([white], device=device))
    us = 0 if white else 1
    for x, perspective in [(x_us, us), (x_them, 1 - us)]:
        if sorted(index for index in x[0].tolist() if index < NUM_FEATURES) != sorted(reference_features(row, perspective)):
            feature_errors += 1
    max_error = max(max_error, abs(predict(check_model, row, white) - reference_eval(check_net, row, white)))

log(f"Feature mismatches: {feature_errors}, largest difference between the model and the engine file: {max_error:.6f}")
assert feature_errors == 0, "the features differ from the engine"
assert max_error < 1e-3, "the engine file does not match the model"
selfplay_arrays = load_selfplay(selfplay_pattern)
board = unpack(torch.from_numpy(selfplay_arrays[0][:1000]).to(device))
assert int((board == 6).sum(dim=1).min()) == 1 and int((board == 12).sum(dim=1).min()) == 1, "every self-play position needs exactly one king per side"
log("Quick check passed")

## Data

In [ ]:
log(f"Start on {device}, streaming data from Lichess with {num_workers} workers...")
lichess_arrays = load_positions(max_positions, max_load_minutes)
num_lichess = len(lichess_arrays[2])
arrays = tuple(np.concatenate([lichess, selfplay]) for lichess, selfplay in zip(lichess_arrays, selfplay_arrays))
del lichess_arrays, selfplay_arrays
num_positions = len(arrays[2])
data_gb = sum(array.nbytes for array in arrays) / 1e9
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if device.type == "cuda" else 0
data_device = device if data_gb < gpu_data_fraction * gpu_gb else torch.device("cpu")
data = tuple(torch.from_numpy(array).to(data_device) for array in arrays)
del arrays
log(f"Data: {num_positions:,} positions ({num_lichess:,} Lichess, {num_positions - num_lichess:,} self-play), {data_gb:.1f} GB, kept on {data_device}")

perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).int().to(data_device)
val_idx = perm[:val_size]
test_idx = perm[val_size:val_size + test_size]
train_idx = perm[val_size + test_size:]
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")
if device.type == "cuda":
    log(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {gpu_gb:.1f} GB")

## Learning rate

Optuna chose 0.0021 in notebook 05 for the same network on almost the same data, so this notebook uses 0.002 and skips the tuning hour.

In [ ]:
log(f"Learning rate: {learning_rate}")

## Final training

In [ ]:
deadline = notebook_start + total_hours * 3600 - reserve_minutes * 60

speed = measure_speed(learning_rate, data, train_idx)
epoch_seconds = 1.1 * len(train_idx) / speed
num_epochs = max(1, min(max_epochs, int((deadline - time.time()) / epoch_seconds)))
log(f"Speed: {speed:,.0f} positions/s, about {epoch_seconds / 60:.0f} min per epoch, so {num_epochs} epochs fit in the time budget")

model = build_model(num_hidden)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=learning_rate / 100)
weights_path = os.path.join(out_dir, "nnue_weights.npz")
engine_path = os.path.join(out_dir, "nnue.bin")
print(model)

history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
best_val_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    if epoch > 0 and time.time() + epoch_seconds > deadline:
        log("Stopping early: not enough time left for another epoch")
        break

    current_lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, optimizer, data, train_idx)
    val_loss, val_mae = eval_model(model, data, val_idx)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["learning_rate"].append(current_lr)
    log(f"Epoch {epoch + 1}/{num_epochs}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        np.savez(weights_path, **model_weights(model))
        export_model(model, engine_path)
        log("Checkpoint saved: nnue_weights.npz and nnue.bin")

## Test

In [ ]:
model.load_state_dict(best_state)
test_loss, test_mae = eval_model(model, data, test_idx)
log(f"Test: loss {test_loss:.4f}, MAE {test_mae:.4f}")

## Sanity check and piece values

The piece values are measured like this: the evaluation of a position where black misses one piece, minus the evaluation of the same position with all pieces. For comparison, v11 gave: opening pawn +64, knight +363, bishop +417, rook +446, queen +667; endgame pawn +165, 2 pawns +368, rook +923.

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white a bishop down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK1NR w KQkq - 0 1",
    "white a queen down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "rook against 3 pawns, white to move (about +200)": "1nbqkbnr/pppppppp/8/8/8/8/3PPPPP/RNBQKBNR w KQk - 0 1",
    "lost game 1, black a bishop down (Stockfish -639)": "2r3k1/5p1p/bq2p1p1/pN1pP3/P2P2n1/RP1Q1NP1/5PBP/6K1 b - - 0 22",
    "lost game 2, black to move (Stockfish -387)": "2krr3/1pp2p1p/p1nb3q/3p4/Q1PP2pN/3bP1P1/P2N1PBP/R2R2K1 b - - 0 16",
}

engine_net = read_nnue(engine_path)
sanity = {}
for name, fen in test_fens.items():
    row, white = parse_fen(fen)
    raw = predict(model, row, white)
    engine_raw = reference_eval(engine_net, row, white)
    sanity[name] = {"fen": fen, "raw": raw}
    log(f"{name}: {raw / CP_SCALE:+.0f} cp for the side to move (engine file: {engine_raw / CP_SCALE:+.0f} cp)")


material_tests = {
    "opening": ("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1", {
        "pawn": "rnbqkbnr/1ppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "knight": "r1bqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "bishop": "rn1qkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "rook": "1nbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQk - 0 1",
        "queen": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    }),
    "endgame": ("r3k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1", {
        "pawn": "r3k3/ppp2pp1/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "2 pawns": "r3k3/ppp2p2/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "rook": "4k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
    }),
}

piece_values = {}
for phase, (start_fen, missing) in material_tests.items():
    start_value = predict(model, *parse_fen(start_fen))
    for name, fen in missing.items():
        value = (predict(model, *parse_fen(fen)) - start_value) / CP_SCALE
        piece_values[f"{phase}: black without {name}"] = value
        log(f"{phase}, black without {name}: {value:+.0f} cp for white")

params = {
    "num_hidden": num_hidden,
    "activation": "screlu",
    "learning_rate": learning_rate,
    "wdl_weight": wdl_weight,
    "num_selfplay": num_positions - num_lichess,
    "num_positions": num_positions,
    "num_lichess": num_lichess,
    "min_depth": min_depth,
    "num_epochs": len(history["train_loss"]),
    "positions_per_second": speed,
}
with open(os.path.join(out_dir, "nnue_selfplay.json"), "w", encoding="utf-8") as f:
    json.dump({"params": params, "history": history, "test_loss": test_loss, "test_mae": test_mae, "sanity": sanity, "piece_values": piece_values}, f, indent=2)
log("Saved nnue_selfplay.json")

## Plots

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["learning_rate"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_selfplay_curves.png"), bbox_inches="tight")
plt.show()

In [ ]:
log(f"Done. Download nnue.bin for the engine. Files: {sorted(os.listdir(out_dir))}")